# Extract all the relevant data of all the scans

Wrestle with all the log files of all the scans.
We double-check all scanning and reconstruction parameters to look for inconsistencies to be corrected.
At the end we generate some helping files which we need for collaboration.

First set up the notebook with some imports and defaults.

In [ ]:
# Load the python modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
import skimage
from tqdm import notebook

In [ ]:
# Load our own log file parsing code
from BrukerSkyScanLogfileRuminator.parsing_functions import *

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
from dask.distributed import Client
client = Client()

In [ ]:
client

In [ ]:
print('You can see what DASK is doing at "http://localhost:%s/status"' % client.scheduler_info()['services']['dashboard'])

In [ ]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
plt.rcParams['figure.figsize'] = (16, 9)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 200

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
local = True
if local:
    # Load the log files from the repository subfolder.
    # Then we cannot 
    Root = os.path.join(os.getcwd(), 'logfiles')
else:
    Root = os.path.join('/home/habi/research_storage_ben/microCT_Stickleback/')
print('We are loading all the data from %s' % Root)

We generate some output in this notebook.
To make all the data completely reproducible, save the output to a directory named according to the current `git` hash of the repository.

In [ ]:
def get_git_hash():
    '''
    Get the current git hash from the repository.
    Based on http://stackoverflow.com/a/949391/323100 and
    http://stackoverflow.com/a/18283905/323100
    '''
    from subprocess import Popen, PIPE
    import os
    gitprocess = Popen(['git',
                        '--git-dir',
                        os.path.join(os.getcwd(), '.git'),
                        'rev-parse',
                        '--short',
                        '--verify',
                        'HEAD'],
                       stdout=PIPE)
    (output, _) = gitprocess.communicate()
    return output.strip().decode("utf-8")

In [ ]:
# Make directory for output
OutPutDir = os.path.join(os.getcwd(), 'Output', get_git_hash())
print('We are saving all the output to %s' % OutPutDir)
os.makedirs(OutPutDir, exist_ok=True)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files, unsorted but fast
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

The notebook might not be running locally on our machines, but on Binder.
There, the user has no access to the log files, so we fail back to a local copy of them.
This also means that no reconstructions are available, und we thus cannot count them.
We thus set a variable which skips looking for parameters related to the reconstructions.

In [ ]:
if not len(Data):
    # Our dataframe is empty.
    # We might be running on Binder, e.g. load the logfiles from the subfolder in this repository
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    print('You are most probably running the notebook on binder.')
    print('And thus do not have access to the log files on the research storage')
    print('We are using a "local" copy of the data in the `logfiles` subfolder')
    print('This gives correct, but possibly outdated results...')
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    # Change root folder
    Root = 'logfiles'
    # Load log files again
    Data['LogFile'] = [f for f in sorted(glob.glob(os.path.join(Root, '**', '*.log'),
                                                   recursive=True),
                                         key=os.path.getmtime)]
    running_on_binder = True
else:
    running_on_binder = False

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]
Data['FolderShort'] = [f[len(Root)+1:] for f in Data['Folder']]

In [ ]:
if not running_on_binder:
    # Check for samples which are not yet reconstructed
    for c, row in Data.iterrows():
        # Iterate over every 'proj' folder
        if 'proj' in row.Folder:
            if 'TScopy' not in row.Folder and 'PR' not in row.Folder:
                # If there's nothing with 'rec*' on the same level, then tell us
                if not glob.glob(row.Folder.replace('proj', 'rec')):
                    # print(glob.glob(row.Folder.replace('proj', 'rec')))
                    print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])

In [ ]:
# Get rid of all log files we do not want to talk about in the manuscript
for c, row in Data.iterrows():
    if 'PilotScans' in row.Folder:  # The pilot scans are mentioned, but not interesting
        Data.drop([c], inplace=True)    
    elif 'rec' not in row.Folder:  # We only want to look at 'rec' folders
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Do not look at the folders with the extracted regions
        Data.drop([c], inplace=True)
# Reset dataframe index
Data = Data.reset_index(drop=True)

In [ ]:
Data.head()

In [ ]:
# Generate us some meaningful colums
Data['Bucket'] = [l[len(Root) + 1:].split(os.sep)[0] for l in Data['LogFile']]
Data['Scan'] = ['.'.join(l[len(Root) + 1:].split(os.sep)[1:-1]) for l in Data['LogFile']]

In [ ]:
Data.head()

In [ ]:
# Get parameters related to scan from logfiles
Data['Scan date'] = [scandate(log) for log in Data['LogFile']]
Data['Scanner'] = [scanner(log) for log in Data['LogFile']]
Data['Voltage'] = [voltage(log) for log in Data['LogFile']]
Data['Current'] = [current(log) for log in Data['LogFile']]
Data['Filter'] = [whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [exposuretime(log) for log in Data['LogFile']]
Data['Averaging'] = [averaging(log) for log in Data['LogFile']]
Data['Number of projections'] = [numproj(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [rotationstep(log) for log in Data['LogFile']]
Data['ThreeSixty'] = [threesixtyscan(log) for log in Data['LogFile']]
Data['Voxelsize'] = [pixelsize(log) for log in Data['LogFile']]
Data['Duration'] = [duration(log) for log in Data['LogFile']]
Data['Stacks'] = [stacks(log) for log in Data['LogFile']]

In [ ]:
# Get parameters related to reconstruction from logfiles
Data['Number of reconstructions'] = [slice_number(log) for log in Data['LogFile']]  # This is the number of reconstructions that NRecon wrote to disk and to the log file. *Not* necessarily the number that may be present on disk. In the InMice project we check if files on disk are the same as filenumber written to log file. But for this manuscript only reading from the log is good enough.
Data['ReconstructionSize'] = [reconstruction_size(log) for log in Data['LogFile']]
Data['Grayvalue'] = [reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [beamhardening(log) for log in Data['LogFile']]
Data['ROI'] = [region_of_interest(log) for log in Data['LogFile']]
Data['NRecon'] = [nreconversion(log)[1] for log in Data['LogFile']]

In [ ]:
Data.head()

In [ ]:
Data.NRecon.unique()

In [ ]:
numpy.sort(Data.Duration.unique())

In [ ]:
# Quickly look at all the different values in our dataframe
DoNotWant = {'LogFile',
             'Folder',
             'FolderShort',
             'Bucket',
             'Scan date'}
for column in Data.columns:
    if column not in DoNotWant:
        print(column, numpy.sort(Data[column].unique()))

In [ ]:
# Sort dataframe by scan date
Data.sort_values(by=['Scan date'], inplace=True)
# Reset dataframe index
Data = Data.reset_index(drop=True)

In [ ]:
Data.head()

In [ ]:
# How many scans did we do?
print('We performed %s scans in total' % len(Data.Scan))

In [ ]:
Data['Total Duration'] = [st * stk for st, stk in zip(Data['Duration'], Data['Stacks'])]

In [ ]:
# Show voxelsize per bucket
for c, vs in enumerate(sorted(Data.Voxelsize.unique())):
    print('-----vs: %s-----' % vs)
    print(Data[Data.Voxelsize == vs][['Bucket', 'Scan', 'Voxelsize']])

In [ ]:
Data.Filter.unique()

In [ ]:
sorted(Data.Voltage.unique())

In [ ]:
sorted(Data.Current.unique())

In [ ]:
sorted(Data.RingartefactCorrection.unique())

In [ ]:
sorted(Data.BeamHardeningCorrection.unique())

In [ ]:
# Search for Fish IDs in `rec_regions`, to show how many *fish* we actually scanned
# In the BucketSeparator-Script we wrote out log files for each extraction
# Above, we dropped all these log files, so we have to construct the folder name again :)
Data['FishRegionsFolder'] = None
for c, row in Data.iterrows():
    # Search a folder up from row.Folder, which is the 'rec' folder
    for root, dirs, files in os.walk(os.path.dirname(row.Folder)):
        # "os.walk returns a 3-tuple containing the directory path, a list of sub-directories, and a list of file names for each directory it visits"
        # So let's look at the 'dirs' that end with 'rec_regions'
        # Then we save that folder into the dataframe
        for directory in dirs:
            if directory.endswith('rec_regions'):
                Data.at[c, 'FishRegionsFolder'] = os.path.join(os.path.dirname(row.Folder), directory)

In [ ]:
# Now that we have the correct folder, we can 'extract' the Fish IDs by simply listing the folders
# This *only* works in the 'logfiles' subfolder, on Bens research storage the 'rec_regions' folder also contains a bunch of other directories with .zarr files and output from the analysis...
Data['FishIDs'] = None
for c, row in Data.iterrows():
    # Here, a simple listdir is sufficient to pull all the folder names
    Data.at[c, 'FishIDs'] = os.listdir(row.FishRegionsFolder)

In [ ]:
Data[['FolderShort', 'FishIDs']].head()

In [ ]:
# Save 'data' file for https://github.com/habi/sticklebacks-manuscript
# Since the manuscript is in a subfolder, we can simply write the output there
if not running_on_binder:
    Data[['FolderShort', 'Bucket', 'Scan', 'Scanner', 'Scan date',
          'Voxelsize', 'Voltage', 'Current',
          'Filter', 'Exposuretime', 'Averaging',
          'Number of projections', 'ProjectionSize', 'RotationStep', 'ThreeSixty',
          'Duration', 'Stacks', 'Total Duration',
          'Number of reconstructions', 'ReconstructionSize', 'ROI',
          'RingartefactCorrection', 'BeamHardeningCorrection', 'Grayvalue', 'NRecon', 'FishIDs'
          ]].to_csv(os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
                    index=False,
                    header=['Folder', 'Bucket', 'Scan', 'Scanner', 'Scan date',
                            'Voxelsize [μm]', 'Source voltage [kV]', 'Source current [μA]',
                            'Filter', 'Exposure time [ms]', 'Frame averaging',
                            'Number of projections', 'Projection size [px]', 'Rotation step [°]', '360° scan',
                            'Scan duration [s]', 'Stacked scans', 'Total scan duration [s]',                                     
                            'Number of reconstructions', 'Reconstruction size [px]', 'Region of interest for reconstruction',
                            'Ring removal correction', 'Beam hardening correction', 'Gray value mapping', 'NRecon version', 'Fish IDs'
                            ])
print('Saved CSV file with all relevant scanning and reconstruction parameters to',
      os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
      'for using as supplementary material in the manuscript')

In [ ]:
# Prepare some sentences for the manuscript

In [ ]:
total_seconds

In [ ]:
# Get an overview of the total scaning time
# Nice output based on https://stackoverflow.com/a/8907407/323100
total_seconds = int(Data['Total Duration'].sum())
days, remainder = divmod(total_seconds, 24 * 60 * 60)
hours, remainder = divmod(remainder, 60 * 60)
minutes, seconds = divmod(remainder, 60)
print('The total scanning duration was %s days, %s hours and %s minutes' % (days, hours, minutes))

In [ ]:
# Give us some numbers for the manuscript
print('In total, we acquired %s projections' % Data['Number of projections'].sum())
print('These were reconstructed into a total of %s reconstructions' % Data['Number of reconstructions'].sum())
print('Which is about %s files per scan (N=%s)' % ((round(Data['Number of reconstructions'].sum() / len(Data))),
                                                   len(Data)))

In [ ]:
# How many fish did we actually scan?
NumFishTotal = len(Data['FishIDs'].explode().unique())
print('We actually scanned N=%s fishe (unique Fish IDs)' % (NumFishTotal))